<a href="https://colab.research.google.com/github/psichen/SPIDER/blob/main/SPIDER_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕷️ SPIDER: Scanning Probe microscopy Image DEnoising and Reconstruction.

Scanning Probe microscopy Image DEnoising and Reconstruction (SPIDER) is a self-supervised image denoising and reconstruction algorithm dedicated for scanning probe micropy (SPM). This notebook gives an intutive and convenient way to perform SPIDER on demo or custom datasets. The SPIDER consists of three modules:

1. hysteresis correction
2. self-supervised denoising
3. (optional) eqvariant reconstruction

If you use SPIDER, please cite the following paper:

- Sichen Pan, Simon Scheuring. "[Self-supervised denoising and restoration method for atomic force microscopy]()" In preparation (2026)

In [ ]:
#@title Initialization
%%time
import subprocess
def run_command(cmd, verbose=False):
  """
  :param cmd: list or str, command
  :param verbose: bool, whether to print output
  """
  print(f"Running: {cmd}")
  process = subprocess.Popen(cmd, shell=isinstance(cmd, str),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1)
  output_lines = []
  for line in process.stdout:
    if verbose:
      print(line, end="")
    output_lines.append(line)
  process.wait()
  output = "".join(output_lines)
  if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, cmd, output=output)

run_command("git clone https://github.com/psichen/SPIDER.git")
run_command("pip install -r ./SPIDER/requirements.txt")

## 1. Hysteresis correction
This section performs subpixel alignment of AFM columns between trace and retrace images to correct piezo hysteresis effect.

### a) file path
- `raw_path` should contain both raw trace and retrace files.
- `data_path` where corrected images will be saved in

In [ ]:
raw_path = "./SPIDER/raw" # @param {type:"string", placeholder:"raw images path"}
trace_file = '20250327_A5_0046_TraceRetrace_0032_50x50nm.tif' # @param {type:"string", placeholder:"trace file name"}
retrace_file = '20250327_A5_0046_TraceRetrace_0032_1ChRet50x50nm.tif' # @param {type:"string", placeholder:"retrace file name"}
data_path = "./SPIDER/datasets" # @param {type:"string", placeholder:"dataset path for training"}

### b) adjustable parameters for hysteresis correction
- `shift`: the shiftment of the attention window
- `window`: the window size
- `trace_left`, `trace_right`, `retrace_left`, `retrace_right`: the boundary ratio used in calculation
- `temperature`: parameter in softmax

In [ ]:
# @title { display-mode: "form", run: "auto" }
shift = 0.04 # @param {type:"slider", min:-0.5, max:0.5, step:0.01}
window = 0.1 # @param {type:"slider", min:0.01, max:0.5, step:0.01}
trace_left = 0.05 # @param {type:"slider", min:0.0, max:0.5, step:0.01}
trace_right = 0.95 # @param {type:"slider", min:0.5, max:1, step:0.01}

# @markdown usually it is not necessary to adjust below parameters
retrace_left = 0. # @param {type:"slider", min:0.0, max:1, step:0.01}
retrace_right = 1. # @param {type:"slider", min:0.0, max:1, step:0.01}
temperature = 0.01 # @param {type:"number"}

from SPIDER.preprocess import data_generator
data_gen = data_generator(raw_path, trace_file, retrace_file, data_path, shift=shift, window=window, tl=trace_left, tr=trace_right, rl=retrace_left, rr=retrace_right, tem=temperature)
data_gen.generate()
data_gen.plot()

In [ ]:
# @markdown save hysteresis-corrected images in the `data_path` { display-mode: "form" }
data_gen.save()

## 2. Self-supervised denoising
This section performs self-supervised denoising step for SPM images. The algorithm is implemented with Distributed Data Parallel (DDP), which allows users to run it with multiple GPUs.

### a) file path
- `checkpoint_path`: path to save checkpoints
- `prediction_path`: path to save predicited images

In [ ]:
checkpoint_path = "./SPIDER/checkpoints" # @param {type:"string", placeholder:"checkpoint path"}
prediction_path = "./SPIDER/predictions" # @param {type:"string", placeholder:"prediction path"}

### b) hyperparameters for denoising
adjustable basic parameters for self-supervised denoising. If scale-up runs or custom modification are needed, running through command lines with more adjustable parameters is recommended.
- `batch_size`
- `epochs`
- `patch_size`: pixel size of which an image is divided into
- `augmentation`
- `ensembles`: ensembels runs
- `world_size`: GPU numbers


In [ ]:
# @markdown hyperparameters for self-supervised denoising
batch_size = 512 # @param {type:"integer"}
iters = 200 # @param {type:"integer"}
patch_size = 64 # @param {type:"integer"}
augmentation = True # @param {type:"boolean"}
ensembles = 3 # @param {type:"integer"}
world_size = 1 # @param {type:"integer"}


In [ ]:
# @markdown run SPIDER
run_command(f"python ./SPIDER/trainer_ddp.py -dp {data_path} -cp {checkpoint_path} -b {batch_size} -it {iters} -ps {patch_size} {'-a' if augmentation else ''} -n {ensembles} -ws {world_size}", verbose=True)
run_command(f"python ./SPIDER/predictor_ddp.py -cp {checkpoint_path} -pp {prediction_path} -dp {data_path} -b {batch_size} -ps {patch_size} -ws {world_size}", verbose=True)
run_command(f"python ./SPIDER/postprocess.py -pp {prediction_path} -dp {data_path}", verbose=True)

In [ ]:
# @markdown Visualization of the results
import plotly.express as px
from skimage import io
import numpy as np

img = io.imread(f'{prediction_path}/output_last_pix.tif')
trace = io.imread(f'{prediction_path}/trace_pix.tif')
retrace = io.imread(f'{prediction_path}/retrace_pix.tif')
imgs = np.concatenate([trace[np.newaxis,...], img[np.newaxis,...], retrace[np.newaxis,...]], axis=0)
fig = px.imshow(imgs, facet_col=0, animation_frame=1, labels=dict(animation_frame="frame"), color_continuous_scale='gray', zmin=np.min(img), zmax=np.max(img))
fig.show()

## 3. Equivariant restoration of SPM images with SPIDER
This section provides an option to restore down-sampled images based on the spatial equivariance of samples. The algorithm is implemented with Distributed Data Parallel (DDP), which allows users to run it with multiple GPUs.

### a) file path
- `data_path`: images with the name "input.tif"
- `checkpoint_path`: path to save checkpoints
- `prediction_path`: path to save predicted images

In [ ]:
data_path = "./SPIDER/equivariant/datasets" # @param {type:"string", placeholder:"dataset path"}
checkpoint_path = "./SPIDER/equivariant/checkpoints" # @param {type:"string", placeholder:"checkpoint path"}
prediction_path = "./SPIDER/equivariant/predictions" # @param {type:"string", placeholder:"prediction path"}

### b) hyperparameters for equivariant restoration
adjustable basic parameters for equivariant restoration. If scale-up runs or custom modification are needed, running through command lines with more adjustable parameters is recommended.
- `batch_size`
- `epochs`
- `patch_size`: pixel size of which an image is divided into
- `weight`: loss weight between consistency and equivariance (0: pure consistency; 1: pure equivariance)
- `scale`: upscale factor
- `augmentation`
- `world_size`: GPU numbers


In [ ]:
# @markdown hyperparameters for equivariant restoration
batch_size = 32 # @param {type:"integer"}
epochs = 20 # @param {type:"integer"}
patch_size = 64 # @param {type:"integer"}
weight = 0.9 # @param {type:"slider", min:0.0, max:1.0, step:0.01}
scale = 2 # @param {type:"integer"}
augmentation = True # @param {type:"boolean"}
world_size = 1 # @param {type:"integer"}


In [ ]:
# @markdown run equivariant restoration
run_command(f"python ./SPIDER/equivariant/trainer_ddp.py -dp {data_path} -cp {checkpoint_path} -w {weight} -s {scale} -b {batch_size} -e {epochs} -ps {patch_size//scale} {patch_size} {'-a' if augmentation else ''} -ws {world_size}", verbose=True)
run_command(f"python ./SPIDER/equivariant/predictor_ddp.py -cp {checkpoint_path} -pp {prediction_path} -dp {data_path} -s {scale} -b {batch_size} -ps {patch_size//scale} {patch_size} -ws {world_size}", verbose=True)


In [ ]:
# @markdown Visualization of the results
import plotly.express as px
from skimage import io
import numpy as np

outputs = io.imread(f'{prediction_path}/ensemble0/last.tif')
inputs = io.imread(f'{data_path}/input.tif')
inputs = np.repeat(inputs, scale, axis=1)
imgs = np.concatenate([inputs[np.newaxis,...], outputs[np.newaxis,...]], axis=0)
fig = px.imshow(imgs, facet_col=0, animation_frame=1, labels=dict(animation_frame="frame"), color_continuous_scale='gray', zmin=np.min(imgs), zmax=np.max(imgs))
fig.show()